### 🛠️ Initialize Notebook Variables

**Only modify entries under _USER CONFIGURATION_.**

In [ ]:
import json
import utils
from typing import List

from apimtypes import API, APIM_SKU, APIOperation, GET_APIOperation, HTTP_VERB, INFRASTRUCTURE, NamedValue, PolicyFragment, Region
from console import print_error, print_ok
from azure_resources import get_infra_rg_name

# ------------------------------
#    USER CONFIGURATION
# ------------------------------

rg_location = Region.EAST_US_2
index       = 1
apim_sku    = APIM_SKU.BASICV2              # Options: 'DEVELOPER', 'BASIC', 'STANDARD', 'PREMIUM', 'BASICV2', 'STANDARDV2', 'PREMIUMV2'
deployment  = INFRASTRUCTURE.SIMPLE_APIM    # Options: see supported_infras below
api_prefix  = 'cors-'                       # ENTER A PREFIX FOR THE APIS TO REDUCE COLLISION POTENTIAL WITH OTHER SAMPLES
tags        = ['dynamic-cors']              # ENTER DESCRIPTIVE TAGS



# ------------------------------
#    SYSTEM CONFIGURATION
# ------------------------------

sample_folder    = 'dynamic-cors'
rg_name          = get_infra_rg_name(deployment, index)
supported_infras = [
    INFRASTRUCTURE.AFD_APIM_PE,
    INFRASTRUCTURE.APIM_ACA,
    INFRASTRUCTURE.APPGW_APIM,
    INFRASTRUCTURE.APPGW_APIM_PE,
    INFRASTRUCTURE.SIMPLE_APIM
]
nb_helper        = utils.NotebookHelper(
    sample_folder,
    rg_name,
    rg_location,
    deployment,
    supported_infras,
    index = index,
    apim_sku = apim_sku
)

# API paths (2 per option, all coexisting side-by-side)
bl_products_path   = f'{api_prefix}bl-products'
bl_analytics_path  = f'{api_prefix}bl-analytics'
opt1_products_path  = f'{api_prefix}opt1-products'
opt1_analytics_path = f'{api_prefix}opt1-analytics'
opt2_products_path  = f'{api_prefix}opt2-products'
opt2_analytics_path = f'{api_prefix}opt2-analytics'
opt3_products_path  = f'{api_prefix}opt3-products'
opt3_analytics_path = f'{api_prefix}opt3-analytics'
opt4_products_path  = f'{api_prefix}opt4-products'
opt4_analytics_path = f'{api_prefix}opt4-analytics'
opt5_products_path  = f'{api_prefix}opt5-products'
opt5_analytics_path = f'{api_prefix}opt5-analytics'
admin_path         = f'{api_prefix}admin'

# Cache key used for CORS origin mapping in Option 3
cors_cache_key = 'corsOriginMapping'

# Allowed origins per API team
products_origins  = ['https://shop.contoso.com', 'https://admin.contoso.com']
analytics_origins = ['https://dashboard.contoso.com']

# JSON mapping for the Named Value (Option 2 API IDs only)
cors_origin_mapping = {
    opt2_products_path:  products_origins,
    opt2_analytics_path: analytics_origins,
}
cors_origin_mapping_json = json.dumps(cors_origin_mapping, separators=(',', ':'))

# JSON mapping for the cache (Option 3 API IDs - single cache entry for all APIs)
cors_cache_mapping = {
    opt3_products_path:  products_origins,
    opt3_analytics_path: analytics_origins,
}
cors_cache_mapping_json = json.dumps(cors_cache_mapping, separators=(',', ':'))

# Per-API cache entries for Option 4 (one cache entry per API)
cors_cache_per_api_mapping = {
    opt4_products_path:  products_origins,
    opt4_analytics_path: analytics_origins,
}

# Per-API Named Value names and JSON values for Option 5
opt5_nv_products_name  = f'CorsOrigins-{opt5_products_path}'
opt5_nv_analytics_name = f'CorsOrigins-{opt5_analytics_path}'

cors_nv_per_api_mapping = {
    opt5_products_path:  {'nv_name': opt5_nv_products_name,  'origins': products_origins},
    opt5_analytics_path: {'nv_name': opt5_nv_analytics_name, 'origins': analytics_origins},
}

# Load operation policies
pol_cors_get = utils.read_policy_xml('cors-get.xml', sample_name = sample_folder)

# Shared operations for all CORS demo APIs
cors_get_op     = GET_APIOperation('Returns mock response with CORS status', pol_cors_get)
cors_options_op = APIOperation('OPTIONS', 'OPTIONS', '/', HTTP_VERB.OPTIONS, 'CORS preflight request')


# ---- Baseline: Native APIM <cors> policy ----

pol_baseline = utils.read_policy_xml('cors-baseline-native.xml', sample_name = sample_folder)

apis_baseline: List[API] = [
    API(bl_products_path, 'Baseline Products', bl_products_path,
        'Products API with native <cors> policy (static origins)',
        pol_baseline, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
    API(bl_analytics_path, 'Baseline Analytics', bl_analytics_path,
        'Analytics API with native <cors> policy (static origins)',
        pol_baseline, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
]


# ---- Option 1: Hard-coded fragment ----

pf_hardcoded_xml = utils.read_policy_xml('pf-dynamic-cors-hardcoded.xml', sample_name = sample_folder)
pf_hardcoded_xml = pf_hardcoded_xml.replace('{products_api_id}', opt1_products_path)
pf_hardcoded_xml = pf_hardcoded_xml.replace('{analytics_api_id}', opt1_analytics_path)

pol_api_opt1 = utils.read_policy_xml('cors-api-policy.xml', sample_name = sample_folder)
pol_api_opt1 = pol_api_opt1.replace('{fragment_id}', 'DynamicCorsHardcoded')

apis_opt1: List[API] = [
    API(opt1_products_path, 'Option 1 Products', opt1_products_path,
        'Products API with hard-coded CORS fragment',
        pol_api_opt1, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
    API(opt1_analytics_path, 'Option 1 Analytics', opt1_analytics_path,
        'Analytics API with hard-coded CORS fragment',
        pol_api_opt1, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
]


# ---- Option 2: Named Values fragment ----

pf_nv_xml = utils.read_policy_xml('pf-dynamic-cors-named-values.xml', sample_name = sample_folder)
pf_nv_xml = pf_nv_xml.replace('{cors_origin_mapping}', '{{CorsOriginMapping}}')

pol_api_opt2 = utils.read_policy_xml('cors-api-policy.xml', sample_name = sample_folder)
pol_api_opt2 = pol_api_opt2.replace('{fragment_id}', 'DynamicCorsNamedValues')

apis_opt2: List[API] = [
    API(opt2_products_path, 'Option 2 Products', opt2_products_path,
        'Products API with Named Values CORS fragment',
        pol_api_opt2, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
    API(opt2_analytics_path, 'Option 2 Analytics', opt2_analytics_path,
        'Analytics API with Named Values CORS fragment',
        pol_api_opt2, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
]


# ---- Option 3: Cache-backed fragment with admin API ----

pf_cached_xml = utils.read_policy_xml('pf-dynamic-cors-cached.xml', sample_name = sample_folder)

pol_api_opt3 = utils.read_policy_xml('cors-api-policy.xml', sample_name = sample_folder)
pol_api_opt3 = pol_api_opt3.replace('{fragment_id}', 'DynamicCorsCached')

pol_admin_load  = utils.read_policy_xml('admin-load-cache.xml', sample_name = sample_folder)
pol_admin_clear = utils.read_policy_xml('admin-clear-cache.xml', sample_name = sample_folder)

cache_key_param = [{'name': 'cacheKey', 'required': True, 'type': 'string', 'description': 'The cache key to operate on'}]

apis_opt3: List[API] = [
    API(opt3_products_path, 'Option 3 Products', opt3_products_path,
        'Products API with cache-backed CORS fragment',
        pol_api_opt3, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
    API(opt3_analytics_path, 'Option 3 Analytics', opt3_analytics_path,
        'Analytics API with cache-backed CORS fragment',
        pol_api_opt3, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
]

api_admin: API = API(
    admin_path, 'Admin', admin_path,
    'Admin API for managing APIM cache entries',
    operations = [
        APIOperation('load-cache', 'Load Cache', '/load-cache/{cacheKey}', HTTP_VERB.POST,
                     'Store a value in the APIM internal cache', pol_admin_load, cache_key_param),
        APIOperation('clear-cache', 'Clear Cache', '/clear-cache/{cacheKey}', HTTP_VERB.POST,
                     'Remove a value from the APIM internal cache', pol_admin_clear, cache_key_param),
    ],
    tags = tags,
    subscriptionRequired = True,
)


# ---- Option 4: Cache-backed fragment with per-API cache entries ----

pf_cached_per_api_xml = utils.read_policy_xml('pf-dynamic-cors-cached-per-api.xml', sample_name = sample_folder)

pol_api_opt4 = utils.read_policy_xml('cors-api-policy.xml', sample_name = sample_folder)
pol_api_opt4 = pol_api_opt4.replace('{fragment_id}', 'DynamicCorsCachedPerApi')

apis_opt4: List[API] = [
    API(opt4_products_path, 'Option 4 Products', opt4_products_path,
        'Products API with per-API cache-backed CORS fragment',
        pol_api_opt4, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
    API(opt4_analytics_path, 'Option 4 Analytics', opt4_analytics_path,
        'Analytics API with per-API cache-backed CORS fragment',
        pol_api_opt4, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
]


# ---- Option 5: Per-API Named Value fragment ----

pf_nv_per_api_xml = utils.read_policy_xml('pf-dynamic-cors-named-values-per-api.xml', sample_name = sample_folder)

pol_api_nv = utils.read_policy_xml('cors-api-policy-named-values.xml', sample_name = sample_folder)

pol_api_opt5_products = pol_api_nv.replace('{allowed_origins_nv}', f'{{{{{opt5_nv_products_name}}}}}')

pol_api_opt5_analytics = pol_api_nv.replace('{allowed_origins_nv}', f'{{{{{opt5_nv_analytics_name}}}}}')

apis_opt5: List[API] = [
    API(opt5_products_path, 'Option 5 Products', opt5_products_path,
        'Products API with per-API Named Value CORS fragment',
        pol_api_opt5_products, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
    API(opt5_analytics_path, 'Option 5 Analytics', opt5_analytics_path,
        'Analytics API with per-API Named Value CORS fragment',
        pol_api_opt5_analytics, [cors_get_op, cors_options_op], tags, subscriptionRequired = False),
]


# ---- Shared: Outbound CORS fragment (used by all options) ----

pf_outbound_xml = utils.read_policy_xml('pf-dynamic-cors-outbound.xml', sample_name = sample_folder)


# ---- Shared: Policy fragments and Named Values ----

pfs: List[PolicyFragment] = [
    PolicyFragment('DynamicCorsHardcoded', pf_hardcoded_xml, 'Dynamic CORS - hard-coded API-to-origins mapping.'),
    PolicyFragment('DynamicCorsNamedValues', pf_nv_xml, 'Dynamic CORS - origins loaded from a Named Value.'),
    PolicyFragment('DynamicCorsCached', pf_cached_xml, 'Dynamic CORS - origins loaded from APIM internal cache.'),
    PolicyFragment('DynamicCorsCachedPerApi', pf_cached_per_api_xml, 'Dynamic CORS - origins loaded from per-API APIM internal cache entries.'),
    PolicyFragment('DynamicCorsNvPerApi', pf_nv_per_api_xml, 'Dynamic CORS - origins passed via per-API Named Value context variable.'),
    PolicyFragment('DynamicCorsOutbound', pf_outbound_xml, 'Dynamic CORS - outbound Access-Control-Allow-Origin header for actual responses.'),
]

nvs: List[NamedValue] = [
    NamedValue('CorsOriginMapping', cors_origin_mapping_json, False),
    NamedValue(opt5_nv_products_name, json.dumps(products_origins, separators=(',', ':')), False),
    NamedValue(opt5_nv_analytics_name, json.dumps(analytics_origins, separators=(',', ':')), False),
]

# Combine all APIs for a single deployment
all_apis = apis_baseline + apis_opt1 + apis_opt2 + apis_opt3 + apis_opt4 + apis_opt5 + [api_admin]


print_ok('Notebook initialized')

### 🚀 Deploy All Options

Deploys all thirteen APIs side-by-side in a single operation alongside all five policy fragments and the Named Values:

| Option | APIs | CORS Mechanism |
| ------ | ---- | -------------- |
| **Baseline** | `cors-bl-products`, `cors-bl-analytics` | Native `<cors>` policy with static origins |
| **Option 1** | `cors-opt1-products`, `cors-opt1-analytics` | `DynamicCorsHardcoded` policy fragment |
| **Option 2** | `cors-opt2-products`, `cors-opt2-analytics` | `DynamicCorsNamedValues` policy fragment |
| **Option 3** | `cors-opt3-products`, `cors-opt3-analytics` | `DynamicCorsCached` policy fragment (single cache entry) |
| **Option 4** | `cors-opt4-products`, `cors-opt4-analytics` | `DynamicCorsCachedPerApi` policy fragment (per-API cache entries) |
| **Option 5** | `cors-opt5-products`, `cors-opt5-analytics` | `DynamicCorsNvPerApi` policy fragment (per-API Named Values) |
| **Admin** | `cors-admin` | Cache management (load + clear) for Options 3 & 4 |

This lets you inspect and compare all six options without redeployment.

In [ ]:
import sys
from pathlib import Path

# Make the sample-local runtime helpers importable from independently run test cells.
dynamic_cors_sample_dir = Path(utils.determine_policy_path('dynamic_cors_helpers.py', sample_folder)).parent
if str(dynamic_cors_sample_dir) not in sys.path:
    sys.path.insert(0, str(dynamic_cors_sample_dir))

results_path = dynamic_cors_sample_dir / 'dynamic-cors-results.local.json'

utils.enable_module_autoreload('dynamic_cors_helpers')

# Deploy all APIs across all options, plus policy fragments and Named Values
bicep_parameters = {
    'apis'            : {'value': [api.to_dict() for api in all_apis]},
    'policyFragments' : {'value': [pf.to_dict() for pf in pfs]},
    'namedValues'     : {'value': [nv.to_dict() for nv in nvs]},
}

output = nb_helper.deploy_sample(bicep_parameters)
deployment_context = nb_helper.get_deployment_context(output)

apim_name        = deployment_context.apim_name
apim_gateway_url = deployment_context.apim_gateway_url
apim_apis        = deployment_context.apis

admin_subscription_key = next(
    (api.get('subscriptionPrimaryKey') for api in apim_apis if api.get('name') == admin_path),
    None,
)

print_ok('Deployment completed successfully - all 13 APIs deployed side-by-side')

### ✅ Test Baseline: Native APIM `<cors>` Policy

The baseline APIs use APIM's built-in `<cors>` policy with a static origin list: `https://shop.contoso.com` and `https://admin.contoso.com`.

| API | Origin | Expected Result |
| --- | ------ | --------------- |
| **Baseline Products** | `https://shop.contoso.com` | PASS - origin is in the static list |
| **Baseline Products** | `https://admin.contoso.com` | PASS - origin is in the static list |
| **Baseline Products** | `https://unauthorized.contoso.net` | PASS (rejected) - origin is not in the list |
| **Baseline Analytics** | `https://dashboard.contoso.com` | **FAIL** - the dashboard origin is not in the static list |

This demonstrates the core limitation: the native `<cors>` policy cannot provide **per-API origin control**.

In [ ]:
from apimtesting import ApimTesting
from dynamic_cors_helpers import DynamicCorsTestRunner, wait_for_gateway_dns

if 'apim_gateway_url' not in locals():
    print_error('Please run the deployment cell first')
    raise SystemExit(1)

wait_for_gateway_dns(apim_gateway_url)

tests_warmup = ApimTesting('Dynamic CORS - Warm-up Test', sample_folder, deployment)
tests_baseline = ApimTesting('Dynamic CORS - Baseline Tests', sample_folder, deployment)

with DynamicCorsTestRunner(
    deployment,
    rg_name,
    apim_gateway_url,
    results_path,
    ['Warm-up', 'Baseline'],
) as runner:
    # Warm up this cell's connection before measuring the scenario requests.
    warmup_response = runner.session.get(
        f'{runner.endpoint_url}/{bl_products_path}',
        headers={'Origin': 'https://shop.contoso.com'},
        timeout=30,
    )
    warmup_ms = round(warmup_response.elapsed.total_seconds() * 1000, 1)
    runner.track(
        tests_warmup,
        'Warm-up',
        'Gateway warm-up GET - establish reusable HTTP session',
        warmup_response.status_code,
        200,
        warmup_ms,
    )
    tests_warmup.print_summary()
    print_ok('HTTP session warm-up complete')

    baseline = 'Baseline'

    status, headers, elapsed_ms = runner.options_request(bl_products_path, 'https://shop.contoso.com')
    runner.track(tests_baseline, baseline, 'BL Products OPTIONS - shop origin (in static list)', status, 200, elapsed_ms)
    runner.track(
        tests_baseline,
        baseline,
        'BL Products Access-Control-Allow-Origin matches shop origin',
        headers.get('Access-Control-Allow-Origin', ''),
        'https://shop.contoso.com',
    )

    status, headers, elapsed_ms = runner.options_request(bl_products_path, 'https://admin.contoso.com')
    runner.track(tests_baseline, baseline, 'BL Products OPTIONS - admin origin (in static list)', status, 200, elapsed_ms)
    runner.track(
        tests_baseline,
        baseline,
        'BL Products Access-Control-Allow-Origin matches admin origin',
        headers.get('Access-Control-Allow-Origin', ''),
        'https://admin.contoso.com',
    )

    _, headers, elapsed_ms = runner.options_request(bl_products_path, 'https://unauthorized.contoso.net')
    runner.track(
        tests_baseline,
        baseline,
        'BL Products OPTIONS - unauthorized origin has no Access-Control-Allow-Origin',
        'Access-Control-Allow-Origin' in headers,
        False,
        elapsed_ms,
    )

    _, headers, elapsed_ms = runner.options_request(bl_analytics_path, 'https://dashboard.contoso.com')
    runner.track(
        tests_baseline,
        baseline,
        'BL Analytics OPTIONS - dashboard origin MISSING from static list '
        '(no Access-Control-Allow-Origin = EXPECTED LIMITATION)',
        'Access-Control-Allow-Origin' in headers,
        False,
        elapsed_ms,
    )

    tests_baseline.print_summary()

print_ok('Baseline tests complete - limitations of native <cors> demonstrated')

### ✅ Test Option 1: Hard-coded CORS Fragment

The Option 1 APIs use the `DynamicCorsHardcoded` policy fragment with per-API origin mapping embedded in C#. Both APIs now get the correct CORS behaviour for their specific frontends.

| API | Origin | Expected Result |
| --- | ------ | --------------- |
| **Option 1 Products** | `https://shop.contoso.com` | PASS - mapped origin |
| **Option 1 Products** | `https://admin.contoso.com` | PASS - mapped origin |
| **Option 1 Products** | `https://unauthorized.contoso.net` | PASS (rejected) - 403 |
| **Option 1 Analytics** | `https://dashboard.contoso.com` | PASS - per-API mapping works (was blocked in baseline) |
| **Option 1 Analytics** | `https://shop.contoso.com` | PASS (rejected) - 403, shop is not an Analytics origin |

In [ ]:
from apimtesting import ApimTesting
from dynamic_cors_helpers import DynamicCorsTestRunner

if 'apim_gateway_url' not in locals():
    print_error('Please run the deployment cell first')
    raise SystemExit(1)

tests_opt1 = ApimTesting('Dynamic CORS - Option 1 Tests', sample_folder, deployment)

with DynamicCorsTestRunner(deployment, rg_name, apim_gateway_url, results_path, ['Option 1']) as runner:
    runner.run_option_tests(tests_opt1, 'Option 1', opt1_products_path, opt1_analytics_path)

tests_opt1.print_summary()
print_ok('Option 1 tests complete - per-API CORS verified via hard-coded fragment')

### ✅ Test Option 2: Named Values CORS Fragment

The Option 2 APIs use the `DynamicCorsNamedValues` policy fragment, which reads the JSON origin mapping from the `CorsOriginMapping` Named Value. All tests should match Option 1 since the mapping data is identical.

**Advantage over Option 1:** Origins can be updated in the Azure portal or via CLI without redeploying the policy fragment. **Limitation:** APIM Named Values have a **4,096-character limit** per value.

In [ ]:
from apimtesting import ApimTesting
from dynamic_cors_helpers import DynamicCorsTestRunner

if 'apim_gateway_url' not in locals():
    print_error('Please run the deployment cell first')
    raise SystemExit(1)

tests_opt2 = ApimTesting('Dynamic CORS - Option 2 Tests', sample_folder, deployment)

with DynamicCorsTestRunner(deployment, rg_name, apim_gateway_url, results_path, ['Option 2']) as runner:
    runner.run_option_tests(tests_opt2, 'Option 2', opt2_products_path, opt2_analytics_path)

tests_opt2.print_summary()
print_ok('Option 2 tests complete - per-API CORS verified via Named Values fragment')

### ✅ Test Option 3: Cache-backed CORS Fragment

Option 3 uses the APIM internal cache to store origin mappings. The cache must be loaded via the admin API before any CORS evaluation can succeed.

**Step 1 - Clear the cache:** Call `POST /cors-admin/clear-cache/corsOriginMapping` to ensure any stale cache entry from a previous run is removed.

**Step 2 - Verify fail-closed behaviour:** With the cache empty, both Option 3 APIs must return `503 Service Unavailable`.

**Step 3 - Load the cache:** Call `POST /cors-admin/load-cache/corsOriginMapping` with the JSON origin mapping.

**Step 4 - Run CORS tests:** After the cache is loaded, Option 3 should behave identically to Options 1 and 2.

> **Note:** The APIM internal cache is shared across the entire instance. The TTL is set to effectively infinite (`2147483647` seconds). Re-calling the load endpoint replaces the previous cache entry. For isolated, high-scale, or multi-region deployments, an external Azure Cache for Redis can be used instead.

> **Security:** The admin API is protected by a subscription key only for simplicity in this lab. In production, add `validate-azure-ad-token` or `validate-jwt` to the admin API's inbound policy. See the [authX](../authX/) and [authX-pro](../authX-pro/) samples for patterns.

In [ ]:
import json

from apimtesting import ApimTesting
from dynamic_cors_helpers import DynamicCorsTestRunner

if 'apim_gateway_url' not in locals():
    print_error('Please run the deployment cell first')
    raise SystemExit(1)

tests_opt3 = ApimTesting('Dynamic CORS - Option 3 Tests', sample_folder, deployment)
admin_headers = {'api-key': admin_subscription_key}

with DynamicCorsTestRunner(deployment, rg_name, apim_gateway_url, results_path, ['Option 3']) as runner:
    print('Step 1: Clearing CORS cache via admin API ...\n')
    clear_response, clear_ms = runner.post(
        f'{admin_path}/clear-cache/{cors_cache_key}',
        headers=admin_headers,
    )
    runner.track(
        tests_opt3,
        'Option 3',
        'OPT3 Admin POST /clear-cache - 200 OK',
        clear_response.status_code,
        200,
        clear_ms,
    )
    print(f'  Clear response: {clear_response.text}')

    print('\nStep 2: Verifying 503 when CORS cache is not initialized ...\n')
    status, _, elapsed_ms = runner.options_request(opt3_products_path, 'https://shop.contoso.com')
    runner.track(tests_opt3, 'Option 3', 'OPT3 Products OPTIONS - 503 before cache load', status, 503, elapsed_ms)

    status, _, elapsed_ms = runner.options_request(opt3_analytics_path, 'https://dashboard.contoso.com')
    runner.track(tests_opt3, 'Option 3', 'OPT3 Analytics OPTIONS - 503 before cache load', status, 503, elapsed_ms)

    print('\nStep 3: Loading CORS origin cache via admin API ...\n')
    load_response, load_ms = runner.post(
        f'{admin_path}/load-cache/{cors_cache_key}',
        headers={**admin_headers, 'Content-Type': 'application/json'},
        data=cors_cache_mapping_json,
    )
    runner.track(
        tests_opt3,
        'Option 3',
        'OPT3 Admin POST /load-cache - 200 OK',
        load_response.status_code,
        200,
        load_ms,
    )
    load_body = load_response.json()
    runner.track(
        tests_opt3,
        'Option 3',
        'OPT3 Admin load response - correct cache key',
        load_body.get('cacheKey'),
        cors_cache_key,
    )
    print(f'  Load response: {json.dumps(load_body, indent=2)}')

    print('\nStep 4: Running CORS tests after cache load ...\n')
    runner.run_option_tests(tests_opt3, 'Option 3', opt3_products_path, opt3_analytics_path)

tests_opt3.print_summary()
print_ok('Option 3 tests complete - cache-backed CORS verified')

### ✅ Test Option 4: Per-API Cache-backed CORS Fragment

Option 4 uses **per-API cache entries** instead of a single mapping. Each API gets its own cache key (`corsOriginMapping-{apiId}`), so updating one API's origins never touches another API's cache entry.

**Step 1 - Clear and verify fail-closed behaviour:** Remove both per-API cache entries, then confirm both Option 4 APIs return `503 Service Unavailable`.

**Step 2 - Load per-API cache entries:** Call `POST /cors-admin/load-cache/corsOriginMapping-{apiId}` once per API with that API's JSON origin array.

**Step 3 - Run CORS tests:** After the cache entries are loaded, Option 4 should behave identically to Options 1, 2, and 3.

> **Advantage over Option 3:** Each request looks up only its own API's origin array, not the full mapping. This reduces cache-read size, especially when many APIs are registered. Updating one API's origins does not require re-loading the entire mapping.

In [ ]:
import json

from apimtesting import ApimTesting
from dynamic_cors_helpers import DynamicCorsTestRunner

if 'apim_gateway_url' not in locals():
    print_error('Please run the deployment cell first')
    raise SystemExit(1)

tests_opt4 = ApimTesting('Dynamic CORS - Option 4 Tests', sample_folder, deployment)
admin_headers = {'api-key': admin_subscription_key}

with DynamicCorsTestRunner(deployment, rg_name, apim_gateway_url, results_path, ['Option 4']) as runner:
    print('Step 1: Clearing per-API cache entries and verifying fail-closed behaviour ...\n')
    for api_id in cors_cache_per_api_mapping:
        cache_key = f'corsOriginMapping-{api_id}'
        clear_response, clear_ms = runner.post(
            f'{admin_path}/clear-cache/{cache_key}',
            headers=admin_headers,
        )
        runner.track(
            tests_opt4,
            'Option 4',
            f'OPT4 Admin POST /clear-cache/{cache_key} - 200 OK',
            clear_response.status_code,
            200,
            clear_ms,
        )

    status, _, elapsed_ms = runner.options_request(opt4_products_path, 'https://shop.contoso.com')
    runner.track(tests_opt4, 'Option 4', 'OPT4 Products OPTIONS - 503 before cache load', status, 503, elapsed_ms)

    status, _, elapsed_ms = runner.options_request(opt4_analytics_path, 'https://dashboard.contoso.com')
    runner.track(tests_opt4, 'Option 4', 'OPT4 Analytics OPTIONS - 503 before cache load', status, 503, elapsed_ms)

    print('\nStep 2: Loading per-API CORS origin cache entries via admin API ...\n')
    load_headers = {**admin_headers, 'Content-Type': 'application/json'}

    for api_id, origins in cors_cache_per_api_mapping.items():
        cache_key = f'corsOriginMapping-{api_id}'
        load_response, load_ms = runner.post(
            f'{admin_path}/load-cache/{cache_key}',
            headers=load_headers,
            data=json.dumps(origins, separators=(',', ':')),
        )
        runner.track(
            tests_opt4,
            'Option 4',
            f'OPT4 Admin POST /load-cache/{cache_key} - 200 OK',
            load_response.status_code,
            200,
            load_ms,
        )

        load_body = load_response.json()
        runner.track(
            tests_opt4,
            'Option 4',
            f'OPT4 Admin load response - correct cache key ({api_id})',
            load_body.get('cacheKey'),
            cache_key,
        )
        print(f'  {api_id}: {json.dumps(load_body, indent=2)}')

    print('\nStep 3: Running CORS tests after per-API cache load ...\n')
    runner.run_option_tests(tests_opt4, 'Option 4', opt4_products_path, opt4_analytics_path)

tests_opt4.print_summary()
print_ok('Option 4 tests complete - per-API cache-backed CORS verified')

### ✅ Test Option 5: Per-API Named Value CORS Fragment

Option 5 uses **per-API Named Values** to supply origin arrays. Each API's policy sets a context variable from its own Named Value (CorsOrigins-{apiId}), then includes a shared, environment-agnostic fragment.

This mirrors the authX-pro pattern where the caller uses <set-variable> with a {{NamedValue}} reference before <include-fragment>. The fragment itself has no knowledge of where the data comes from.

| API | Origin | Expected Result |
| --- | ------ | --------------- |
| **Option 5 Products** | https://shop.contoso.com | PASS - origin is in the per-API Named Value |
| **Option 5 Products** | https://admin.contoso.com | PASS - origin is in the per-API Named Value |
| **Option 5 Products** | https://unauthorized.contoso.net | PASS (rejected) - 403 |
| **Option 5 Analytics** | https://dashboard.contoso.com | PASS - per-API Named Value works |
| **Option 5 Analytics** | https://shop.contoso.com | PASS (rejected) - 403, shop is not an Analytics origin |

**Advantages over Option 2:** No character-limit concern from a shared mapping object. Each API's origins are independent. Updating one API's Named Value does not affect others. **Advantage over Option 4:** No admin API or cache warm-up required - origins are available immediately at deployment time.

In [ ]:
from apimtesting import ApimTesting
from dynamic_cors_helpers import DynamicCorsTestRunner

if 'apim_gateway_url' not in locals():
    print_error('Please run the deployment cell first')
    raise SystemExit(1)

tests_opt5 = ApimTesting('Dynamic CORS - Option 5 Tests', sample_folder, deployment)

with DynamicCorsTestRunner(deployment, rg_name, apim_gateway_url, results_path, ['Option 5']) as runner:
    runner.run_option_tests(tests_opt5, 'Option 5', opt5_products_path, opt5_analytics_path)

tests_opt5.print_summary()
print_ok('Option 5 tests complete - per-API CORS verified via Named Value context variable')

### 📊 Consolidated Test Results

Aggregates the results from all six test options into a single summary table.

In [ ]:
import pandas as pd
from IPython.display import display

from console import print_error, print_ok
from dynamic_cors_helpers import load_test_results

if 'results_path' not in locals():
    print_error('Please run the deployment cell first')
    raise SystemExit(1)

test_results = load_test_results(results_path)
if not test_results:
    print_error('No Dynamic CORS test results found. Run one or more test cells first.')
    raise SystemExit(1)

df = pd.DataFrame(test_results)
df.index = range(1, len(df) + 1)
df.index.name = '#'
df['Test'] = df['Test'].str.replace(r'^(?:BL|OPTION\d|OPT\d)\s+', '', regex=True)
df['Result'] = df['Result'].map({True: '✅ Pass', False: '❌ Fail'})
df['Duration (ms)'] = df['Duration (ms)'].apply(lambda value: f'{value:.1f}' if pd.notna(value) else 'N/A')

passed = (df['Result'] == '✅ Pass').sum()
failed = (df['Result'] == '❌ Fail').sum()
total = len(df)

# Each option uses explicit dark text so the result table remains readable across themes.
option_styles = {
    'Warm-up':  'background-color: #e5e7eb; color: #111827',
    'Baseline': 'background-color: #dbeafe; color: #1e3a5f',
    'Option 1': 'background-color: #fef3c7; color: #78350f',
    'Option 2': 'background-color: #ede9fe; color: #4c1d95',
    'Option 3': 'background-color: #fce7f3; color: #831843',
    'Option 4': 'background-color: #ccfbf1; color: #134e4a',
    'Option 5': 'background-color: #fef9c3; color: #713f12',
}
style_pass = 'background-color: #d1fae5; color: #065f46; font-weight: 600'
style_fail = 'background-color: #fee2e2; color: #7f1d1d; font-weight: 600'


def highlight_row(row):
    """Colour the row by option and emphasize its result cell."""
    option_css = option_styles.get(row['Option'], 'color: #212529')
    styles = [option_css] * len(row)
    result_index = row.index.get_loc('Result')
    if '✅' in str(row['Result']):
        styles[result_index] = style_pass
    elif '❌' in str(row['Result']):
        styles[result_index] = style_fail
    return styles


styled = (
    df.style
    .apply(highlight_row, axis=1)
    .set_properties(**{
        'text-align': 'left',
        'border': '1px solid #d1d5db',
        'padding': '6px 10px',
    })
    .set_properties(
        subset=['Duration (ms)'],
        **{'text-align': 'right', 'font-variant-numeric': 'tabular-nums'},
    )
    .set_table_styles([
        {
            'selector': 'th',
            'props': [
                ('background-color', '#1e3a5f'),
                ('color', 'white'),
                ('padding', '10px 12px'),
                ('text-align', 'left'),
                ('border', '1px solid #d1d5db'),
                ('font-weight', '600'),
            ],
        },
        {
            'selector': 'caption',
            'props': [
                ('font-size', '15px'),
                ('font-weight', 'bold'),
                ('padding', '10px 0'),
                ('color', '#1e3a5f'),
            ],
        },
        {
            'selector': 'td',
            'props': [('border-bottom', '1px solid #e5e7eb')],
        },
    ])
    .set_caption(f'Total: {total}  |  Passed: {passed}  |  Failed: {failed}')
)

display(styled)
print_ok('All done!')